# Phase 2 · Model 1 · Adaptation and character-level CRF

The full fine-tuning, bottleneck adapter, and LoRMA+ runs start from the same Phase 1 decoder checkpoint. They share the document split, bidirectional character head, masked convolutions, and BIO decoding. The CRF experiment uses the adapter configuration and learns character-level BIO transition scores in a separate run.

The methods save under `phase2_model1_comparison/`. The CRF run saves under `model1_adapters_crf_v1/`. Each method selects its own best validation checkpoint.

LoRMA+ applies the multiplicative update from Eq. 8 of [Bihany et al.](https://arxiv.org/html/2506.07621v1) to Q, K, V, and the attention output projection. Bottleneck adapters insert two trainable residual modules per block. Both methods freeze the original backbone.

## A1. Required files and environment

- Attach the Model 1 Phase 1 run folder with `best.pt`, `run_config.json`, and `tokenizer.json`.
- Attach `evaluate_phase2.py` and the official `Phase_2_LNER` dataset. To download the dataset in Kaggle, enable Internet and set the `HF_TOKEN` or `HF` secret.
- Enable a GPU. For a saved comparison, set `RESUME_COLLECTION` to the restored run folder. Set `REFERENCE_SPLIT` if you are reproducing a specific validation split.

In [1]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','huggingface-hub','matplotlib'],check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'numpy', 'huggingface-hub', 'matplotlib'], returncode=0)

In [2]:
import copy, gc, hashlib, heapq, json, math, os, random, re, shutil, time, zipfile
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from IPython.display import display, HTML

SOURCES = ('supreme_court', 'high_courts', 'acts', 'tribunals')
PAD, UNK, BOS, EOS = range(4)
VERSION = 'local-global-causal-v1'

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp, path)

def digest_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024**2), b''): h.update(block)
    return h.hexdigest()

def atomic_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(value, str(path) + '.tmp')
    os.replace(str(path) + '.tmp', path)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(),
                torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda']: torch.cuda.set_rng_state_all(s['cuda'])

from collections import Counter
import bisect


In [3]:
INPUT_ROOT=Path('/kaggle/input')
BASE_RUN=None
BASE_CHECKPOINT=None
DATA_ROOT=None
EVALUATOR_PATH=None
REFERENCE_SPLIT=None
RESUME_COLLECTION=None
COLLECTION=Path('/kaggle/working/phase2_model1_comparison')
RUN_DIR=COLLECTION/'full'
ASSET_DIR=COLLECTION/'assets'
REPO='Exploration-Lab/CS-781-Capstone'
DATA_REVISION=None
COMMON_CONFIG=dict(seed=42,validation_fraction=.15,window=1024,overlap=256,
    epochs=40,effective_batch=4,micro_batch=1,activation_checkpointing=True,
    backbone_lr=5e-5,head_lr=5e-4,min_lr_ratio=.1,warmup_fraction=.06,
    weight_decay=.01,clip=1.,eval_every=1000,save_every=100,log_every=20,
    class_weight_cap=5.,head_dim=128,char_buckets=4096,bi_dropout=.1,
    adapter_dim=64,lorma_rank=16,lorma_alpha=16,max_hours=10.,head_version='model1-bi-conv-v1')
CONFIG=dict(COMMON_CONFIG,method='full')
if RESUME_COLLECTION is not None:
    origin=Path(RESUME_COLLECTION)
    if origin.resolve()!=COLLECTION.resolve():
        if COLLECTION.exists() and any(COLLECTION.iterdir()): raise ValueError('Restore into an empty collection folder')
        shutil.copytree(origin,COLLECTION)
ASSET_DIR.mkdir(parents=True,exist_ok=True)
if BASE_RUN is None:
    candidates=[]
    for path in INPUT_ROOT.rglob('run_config.json'):
        cfg=read_json(path); model=cfg.get('model',{})
        if 'heads' in model and 'layers' in model and 'window' not in model and 'encoder_layers' not in model:
            if (path.parent/'best.pt').is_file() and (path.parent/'tokenizer.json').is_file(): candidates.append(path.parent)
    if len(candidates)!=1: raise ValueError(f'Set BASE_RUN to one Phase-1 model-1 folder: {candidates}')
    BASE_RUN=candidates[0]
BASE_RUN=Path(BASE_RUN)
BASE_CHECKPOINT=Path(BASE_CHECKPOINT) if BASE_CHECKPOINT else BASE_RUN/'best.pt'
if EVALUATOR_PATH is None:
    candidates=list(INPUT_ROOT.rglob('evaluate_phase2.py'))
    if len(candidates)!=1: raise ValueError(f'Set EVALUATOR_PATH: {candidates}')
    EVALUATOR_PATH=candidates[0]
EVALUATOR_PATH=Path(EVALUATOR_PATH)
print('Parent:',BASE_CHECKPOINT)
print('Runs:',COLLECTION)


Parent: /kaggle/input/datasets/acadsiitk/model1-phase2-data/best.pt
Runs: /kaggle/working/phase2_model1_comparison


In [4]:
def get_data_root(explicit=None):
    needed=['Train/fold1-text.json','Train/fold1-gold.json','Train/fold2-text.json',
            'Train/fold2-gold.json','Train/labels.txt','Test/fold3-text.json']
    def valid(root): return all((root/p).is_file() for p in needed)
    if explicit is not None:
        root=Path(explicit)
        if not valid(root): raise FileNotFoundError(root)
        return root
    found=sorted({p.parent.parent for p in INPUT_ROOT.rglob('fold1-text.json') if valid(p.parent.parent)})
    if len(found)==1: return found[0]
    if len(found)>1: raise ValueError(f'Set DATA_ROOT: {found}')
    root=ASSET_DIR/'Phase_2_LNER'
    if valid(root): return root
    token=os.environ.get('HF_TOKEN')
    try:
        from kaggle_secrets import UserSecretsClient
        secrets=UserSecretsClient()
        for name in ('HF_TOKEN','HF'):
            try:
                token=secrets.get_secret(name)
                if token: break
            except Exception: pass
    except ImportError: pass
    from huggingface_hub import HfApi,hf_hub_download
    revision=HfApi().dataset_info(REPO,revision=DATA_REVISION,token=token).sha
    for name in needed:
        path=hf_hub_download(repo_id=REPO,repo_type='dataset',revision=revision,
            filename='Phase_2_LNER/'+name,token=token)
        target=root/name; target.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(path,target)
    write_json(ASSET_DIR/'dataset_revision.json',dict(repo=REPO,revision=revision))
    return root


In [5]:
DATA_ROOT=get_data_root(DATA_ROOT)
print("Data:",DATA_ROOT)


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Data: /kaggle/working/phase2_model1_comparison/assets/Phase_2_LNER


## A2. Tokenizer and pretrained backbone


In [6]:
def split_bpe_pieces(text, max_chars=256):
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


In [7]:
def apply_bpe_merges(piece, merge_ranks):
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


In [8]:
class ScratchByteBPE:
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [9]:
class Attention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)

class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = Attention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


In [10]:
class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)

class ResidualAdapter(nn.Module):
    def __init__(self,w,r):
        super().__init__(); self.down=nn.Linear(w,r); self.up=nn.Linear(r,w)
        nn.init.normal_(self.down.weight,std=.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return x+self.up(F.gelu(self.down(x)))

class LoRMAPlus(nn.Module):
    def __init__(self,linear,rank,alpha,groups=1):
        super().__init__(); self.weight=linear.weight; self.bias=linear.bias
        self.weight.requires_grad_(False)
        if self.bias is not None: self.bias.requires_grad_(False)
        self.groups=groups; self.width=linear.out_features//groups; self.scale=alpha/rank
        assert linear.out_features%groups==0 and 0<rank<=self.width
        self.lorma_A=nn.Parameter(torch.empty(groups,rank,self.width))
        self.lorma_B=nn.Parameter(torch.zeros(groups,self.width,rank))
        nn.init.normal_(self.lorma_A,std=.02)
    def forward(self,x):
        z=F.linear(x,self.weight,None)
        chunks=z.split(self.width,dim=-1)
        result=torch.cat([h+self.scale*F.linear(F.linear(h,self.lorma_A[i]),self.lorma_B[i])
                          for i,h in enumerate(chunks)],dim=-1)
        return result if self.bias is None else result+self.bias

class Model1Backbone(DecoderLM):
    def __init__(self,model_cfg,cfg):
        super().__init__(**model_cfg); self.method=cfg['method']
        self.gradient_checkpointing=cfg['activation_checkpointing']
        if self.method!='full':
            for p in self.parameters(): p.requires_grad_(False)
        if self.method=='adapters':
            self.attention_adapters=nn.ModuleList([ResidualAdapter(model_cfg['width'],cfg['adapter_dim']) for _ in self.blocks])
            self.ff_adapters=nn.ModuleList([ResidualAdapter(model_cfg['width'],cfg['adapter_dim']) for _ in self.blocks])
        elif self.method=='lorma':
            for block in self.blocks:
                block.attn.qkv=LoRMAPlus(block.attn.qkv,cfg['lorma_rank'],cfg['lorma_alpha'],groups=3)
                block.attn.proj=LoRMAPlus(block.attn.proj,cfg['lorma_rank'],cfg['lorma_alpha'])
        elif self.method!='full': raise ValueError('Unknown method')
    def load_base(self,state):
        extra={k for k in self.state_dict() if k.startswith(('attention_adapters.','ff_adapters.')) or '.lorma_' in k}
        if set(state)!=set(self.state_dict())-extra: raise ValueError('Model-1 checkpoint keys do not match')
        result=self.load_state_dict(state,strict=False)
        if set(result.missing_keys)!=extra or result.unexpected_keys: raise ValueError('Incomplete pretrained weights')
    def hidden(self,ids):
        if ids.shape[1]>self.config['context']: raise ValueError('Context exceeded')
        x=self.drop(self.token(ids)+self.position(torch.arange(ids.shape[1],device=ids.device)))
        for i,block in enumerate(self.blocks):
            def layer(z,block=block,i=i):
                if self.method!='adapters': return block(z)[0]
                z=self.attention_adapters[i](z+block.drop(block.attn(block.ln1(z))[0]))
                return self.ff_adapters[i](z+block.drop(block.mlp(block.ln2(z))))
            x=checkpoint(layer,x,use_reentrant=False) if self.training and self.gradient_checkpointing else layer(x)
        return self.final_norm(x),None


In [11]:
TOKENIZER=ScratchByteBPE.from_file(BASE_RUN/'tokenizer.json')
BASE_STATE=torch.load(BASE_CHECKPOINT,map_location='cpu',weights_only=False)
MODEL_CONFIG=copy.deepcopy(BASE_STATE['model_config'])
if set(MODEL_CONFIG)!={'vocab_size','width','layers','heads','context','dropout'}: raise ValueError('Expected model-1 architecture')
if digest_file(BASE_RUN/'tokenizer.json')!=BASE_STATE['tokenizer_sha256']: raise ValueError('Tokenizer mismatch')
if MODEL_CONFIG['vocab_size']!=TOKENIZER.get_vocab_size(): raise ValueError('Vocabulary mismatch')
BASE_EXPOSURE=BASE_STATE.get('tokens',BASE_STATE.get('tokens_completed'))
assert 0<=CONFIG['overlap']<CONFIG['window']<=MODEL_CONFIG['context']
for text in ['भारत ₹100','a  b\n c']:
    assert TOKENIZER.decode(TOKENIZER.encode(text).ids)==text
print('Parent token count:',BASE_EXPOSURE)


Parent token count: 442365850


## B1. Documents, annotations and validation split

Text is kept unchanged. Offsets use Python character indices and exclusive ends; annotated entity text is checked against the corresponding slice when present. Duplicate documents share a split. Exact duplicate text with conflicting annotations stops preparation. The fixed split is chosen using label coverage, before training; it is reused across experiments. Missing validation labels are reported, not omitted from Macro-F1.


In [12]:
def strict_json(path):
    def pairs(items):
        out={}
        for k,v in items:
            if k in out: raise ValueError(f'Duplicate JSON key: {k}')
            out[k]=v
        return out
    return json.loads(Path(path).read_text(encoding='utf-8'),object_pairs_hook=pairs)

def load_corpus(root):
    labels=[x.strip() for x in (root/'Train/labels.txt').read_text(encoding='utf-8-sig').splitlines() if x.strip()]
    if len(labels)!=12 or len(set(labels))!=12: raise ValueError('Expected 12 distinct labels')
    texts={}; gold={}
    for fold in (1,2):
        t=strict_json(root/f'Train/fold{fold}-text.json'); g=strict_json(root/f'Train/fold{fold}-gold.json')
        if not isinstance(t,dict) or not isinstance(g,dict) or set(t)!=set(g): raise ValueError('Training ID mismatch')
        for key,text in t.items():
            if not isinstance(text,str) or not text: raise ValueError(f'Invalid text: {key}')
            spans=[]
            if not isinstance(g[key],list): raise ValueError(f'Expected entity list: {key}')
            for item in g[key]:
                if not isinstance(item,list) or len(item) not in (3,4): raise ValueError(f'Invalid entity: {key}: {item}')
                a,b,label=item[:3]
                if type(a)!=int or type(b)!=int or not 0<=a<b<=len(text) or label not in labels:
                    raise ValueError(f'Invalid span: {key}: {item}')
                if len(item)==4 and text[a:b]!=item[3]: raise ValueError(f'Offset/text mismatch: {key}: {item}')
                spans.append([a,b,label])
            spans.sort()
            if len({tuple(s) for s in spans})!=len(spans): raise ValueError(f'Duplicate annotations: {key}')
            if any(left[1]>right[0] for left,right in zip(spans,spans[1:])):
                raise ValueError(f'Overlapping gold spans in {key}; use an overlapping-span head before training')
            if key in texts and (texts[key]!=text or gold[key]!=spans): raise ValueError(f'Conflicting fold ID: {key}')
            texts[key]=text; gold[key]=spans
    test=strict_json(root/'Test/fold3-text.json')
    if not isinstance(test,dict) or not test or any(not isinstance(t,str) for t in test.values()): raise ValueError('Invalid test documents')
    return texts,gold,test,labels

def make_split(texts,gold,labels,seed,fraction):
    groups=defaultdict(list)
    for key,text in texts.items(): groups[hashlib.sha256(text.encode('utf-8')).hexdigest()].append(key)
    for keys in groups.values():
        if any(gold[k]!=gold[keys[0]] for k in keys): raise ValueError('Duplicate text with conflicting annotations')
    hashes=sorted(groups)
    if len(hashes)<2: raise ValueError('Need at least two distinct documents')
    count=max(1,min(len(hashes)-1,round(len(hashes)*fraction)))
    best=None
    for attempt in range(100):
        order=np.random.default_rng(seed+attempt).permutation(hashes)
        val=sorted(k for h in order[:count] for k in groups[h]); train=sorted(k for h in order[count:] for k in groups[h])
        coverage=[{s[2] for k in ids for s in gold[k]} for ids in (train,val)]
        score=(len(coverage[0]&coverage[1]),len(coverage[0]),len(coverage[1]))
        if best is None or score>best[0]: best=(score,train,val)
    return dict(train=best[1],val=best[2])

def char_targets(text,spans,labels):
    y=np.zeros(len(text),dtype=np.int64); lookup={l:i for i,l in enumerate(labels)}
    for a,b,label in spans:
        k=lookup[label]; y[a]=1+2*k; y[a+1:b]=2+2*k
    return y

def decode_tags(tags,labels):
    spans=[]; start=None; active=None
    for i,value in enumerate(list(tags)+[0]):
        value=int(value); kind=None if value==0 else (value-1)//2
        begins=value>0 and value%2==1
        if start is not None and (kind!=active or begins):
            spans.append([start,i,labels[active]]); start=None
        if kind is not None and start is None: start=i
        active=kind
    return spans


In [13]:
TEXTS,GOLD,TEST_TEXTS,LABELS=load_corpus(DATA_ROOT)
split=make_split(TEXTS,GOLD,LABELS,CONFIG['seed'],CONFIG['validation_fraction'])
if (RUN_DIR/'split.json').exists():
    saved=read_json(RUN_DIR/'split.json')
    if saved!=split: raise ValueError('Validation split changed')
for name,ids in split.items():
    counts=Counter(s[2] for key in ids for s in GOLD[key])
    print(name,len(ids),'documents',dict(counts))
    print('Missing labels:',sorted(set(LABELS)-set(counts)))
for key in TEXTS:
    assert decode_tags(char_targets(TEXTS[key],GOLD[key],LABELS),LABELS)==GOLD[key]

if REFERENCE_SPLIT is not None and read_json(REFERENCE_SPLIT)!=split:
    raise ValueError("Reference split mismatch; check dataset revision and split settings")


train 60 documents {'APP': 352, 'DATE': 1342, 'JUDGE': 216, 'CASENO': 690, 'STAT': 1157, 'RESP': 353, 'AUTH': 903, 'COURT': 954, 'A.COUNSEL': 142, 'R.COUNSEL': 153, 'PREC': 1026, 'WIT': 172}
Missing labels: []
val 10 documents {'APP': 55, 'RESP': 28, 'DATE': 233, 'JUDGE': 29, 'CASENO': 70, 'AUTH': 100, 'STAT': 103, 'PREC': 157, 'A.COUNSEL': 53, 'R.COUNSEL': 31, 'COURT': 59, 'WIT': 15}
Missing labels: []


## B2. Character alignment and overlapping windows

Every character is scored exactly once. Overlap supplies left context; duplicated context characters do not contribute to the loss. Full-document tags are assembled before span decoding, so entities can cross a window boundary. A character whose UTF-8 bytes span several BPE tokens maps to the last overlapping token. Character identity and within-token position distinguish characters sharing a hidden state.


In [14]:
def encode_document(text,tok):
    ids=np.asarray(tok.encode(text).ids,dtype=np.int64)
    byte_lengths=np.asarray([len(tok.token_bytes[int(i)]) for i in ids],dtype=np.int64)
    token_ends=np.cumsum(byte_lengths)
    char_ends=np.cumsum([len(ch.encode('utf-8')) for ch in text],dtype=np.int64)
    if not np.array_equal(np.asarray([sum(byte_lengths)]),np.asarray([len(text.encode('utf-8'))])): raise ValueError('Byte alignment failed')
    if tok.decode(ids)!=text: raise ValueError('Tokenizer round trip failed')
    mapping=np.searchsorted(token_ends,char_ends,side='left')
    positions=np.zeros(len(text),dtype=np.int64)
    first=0
    for i in range(len(text)):
        if i==0 or mapping[i]!=mapping[i-1]: first=i
        positions[i]=min(255,i-first)
    return dict(ids=ids,mapping=mapping,characters=np.asarray([ord(c) for c in text],dtype=np.int64),positions=positions)

class WindowData:
    def __init__(self,texts,keys,tok,cfg,gold=None,labels=None):
        self.docs={}; self.rows=[]; self.cfg=cfg
        for key in keys:
            text=texts[key]; doc=encode_document(text,tok)
            doc['targets']=char_targets(text,gold[key],labels) if gold is not None else np.zeros(len(text),dtype=np.int64)
            self.docs[key]=doc
            previous=0; start=0
            while start<len(doc['ids']):
                end=min(start+cfg['window'],len(doc['ids']))
                a=int(np.searchsorted(doc['mapping'],previous,'left'))
                b=int(np.searchsorted(doc['mapping'],end,'left'))
                if b>a: self.rows.append((key,start,end,a,b))
                previous=end
                if end==len(doc['ids']): break
                start=end-cfg['overlap']
    def __len__(self): return len(self.rows)
    def batch(self,indices,device):
        rows=[self.rows[int(i)] for i in indices]
        nt=max(r[2]-r[1] for r in rows); nc=max(r[4]-r[3] for r in rows)
        x=np.full((len(rows),nt),PAD,dtype=np.int64); mapping=np.full((len(rows),nc),-1,dtype=np.int64)
        chars=np.zeros_like(mapping); positions=np.zeros_like(mapping); y=np.full_like(mapping,-100)
        for i,(key,start,end,a,b) in enumerate(rows):
            doc=self.docs[key]; n=b-a
            x[i,:end-start]=doc['ids'][start:end]
            mapping[i,:n]=doc['mapping'][a:b]-start
            chars[i,:n]=doc['characters'][a:b]%self.cfg['char_buckets']
            positions[i,:n]=doc['positions'][a:b]; y[i,:n]=doc['targets'][a:b]
            assert np.all(mapping[i,:n]>=0)
        return tuple(torch.from_numpy(v).to(device) for v in (x,mapping,chars,positions,y))

def verify_windows(data,texts):
    covered=defaultdict(int)
    for key,start,end,a,b in data.rows:
        assert a==covered[key] and 0<=start<end<=len(data.docs[key]['ids'])
        covered[key]=b
    assert all(covered[k]==len(texts[k]) for k in data.docs)

def make_weights(data,nlabels,cap):
    counts=np.zeros(nlabels,dtype=np.int64)
    for doc in data.docs.values(): counts+=np.bincount(doc['targets'],minlength=nlabels)
    weights=np.ones(nlabels,dtype=np.float32)
    observed=counts>0
    weights[observed]=np.minimum(cap,np.sqrt(counts.max()/counts[observed]))
    weights/=np.sum(weights*counts)/max(1,counts.sum())
    return weights,counts


In [15]:
TRAIN_DATA=WindowData(TEXTS,split['train'],TOKENIZER,CONFIG,GOLD,LABELS)
VAL_DATA=WindowData(TEXTS,split['val'],TOKENIZER,CONFIG,GOLD,LABELS)
verify_windows(TRAIN_DATA,TEXTS); verify_windows(VAL_DATA,TEXTS)
CLASS_WEIGHTS,TAG_COUNTS=make_weights(TRAIN_DATA,1+2*len(LABELS),CONFIG['class_weight_cap'])
print('Windows:',len(TRAIN_DATA),'train,',len(VAL_DATA),'validation')
print('Longest character batch row:',max(b-a for _,_,_,a,b in TRAIN_DATA.rows))


Windows: 529 train, 79 validation
Longest character batch row: 5422


## C. NER head and loss


In [16]:
class LegalNER(nn.Module):
    def __init__(self,model_cfg,nlabels,cfg,weights):
        super().__init__()
        self.backbone=Model1Backbone(model_cfg,cfg)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(cfg['seed']+1000)
            w=model_cfg['width']; d=cfg['head_dim']
            self.bi=nn.TransformerEncoderLayer(
                d_model=w,nhead=model_cfg['heads'],dim_feedforward=2*w,
                dropout=cfg['bi_dropout'],batch_first=True,norm_first=True)
            self.projection=nn.Linear(w,d)
            self.characters=nn.Embedding(cfg['char_buckets'],d)
            self.positions=nn.Embedding(256,d)
            self.mix=nn.Sequential(nn.Conv1d(d,d,9,padding=4),nn.GELU(),nn.Conv1d(d,d,9,padding=4))
            self.head=nn.Sequential(nn.LayerNorm(d),nn.GELU(),nn.Linear(d,nlabels))
            self.register_buffer('class_weights',torch.as_tensor(weights,dtype=torch.float32))

    def forward(self,x,mapping,chars,positions,y=None):
        valid=mapping.ge(0)
        h,_=self.backbone.hidden(x)
        h=self.bi(h,src_key_padding_mask=x.eq(PAD))
        h=self.projection(h)
        h=h.gather(1,mapping.clamp_min(0)[:,:,None].expand(-1,-1,h.shape[-1]))
        mask=valid[:,:,None].to(h.dtype)
        z=(h+self.characters(chars)+self.positions(positions))*mask
        conv_mask=mask.transpose(1,2)
        mixed=F.gelu(self.mix[0](z.transpose(1,2)))*conv_mask
        mixed=self.mix[2](mixed)*conv_mask
        logits=self.head((z+mixed.transpose(1,2))*mask)
        if y is None: return logits
        if not torch.equal(valid,y.ne(-100)): raise ValueError('Character padding mask mismatch')
        return F.cross_entropy(logits.float().flatten(0,1),y.flatten(),
            weight=self.class_weights,ignore_index=-100,reduction='sum').reshape(1)

def make_model(cfg,device):
    mc=copy.deepcopy(MODEL_CONFIG)
    model=LegalNER(mc,1+2*len(LABELS),cfg,CLASS_WEIGHTS)
    model.backbone.load_base(BASE_STATE['model'])
    return model.to(device)

class PrecisionWrapper(nn.Module):
    def __init__(self,model): super().__init__(); self.model=model
    def forward(self,*args):
        with torch.autocast(device_type=args[0].device.type,dtype=torch.float16,enabled=args[0].is_cuda):
            return self.model(*args)

def setup_training(cfg,cpu=False):
    device=torch.device('cpu' if cpu else 'cuda')
    model=make_model(cfg,device)
    groups=[dict(params=[p for p in model.backbone.parameters() if p.requires_grad],lr=cfg['backbone_lr'],base_lr=cfg['backbone_lr']),
            dict(params=[p for n,p in model.named_parameters() if not n.startswith('backbone.')],lr=cfg['head_lr'],base_lr=cfg['head_lr'])]
    optimizer=torch.optim.AdamW(groups,weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=not cpu)
    wrapper=PrecisionWrapper(model)
    if not cpu and torch.cuda.device_count()>1: wrapper=nn.DataParallel(wrapper)
    return model,optimizer,scaler,wrapper,device

def update_batch(model,optimizer,scaler,wrapper,data,indices,cfg,device):
    batch=cfg['micro_batch']*(torch.cuda.device_count() if device.type=='cuda' else 1)
    denominator=sum(float(CLASS_WEIGHTS[data.docs[key]['targets'][a:b]].sum())
                    for key,_,_,a,b in (data.rows[int(i)] for i in indices))
    if denominator<=0: raise ValueError('Empty training batch')
    model.train()
    for retry in range(5):
        optimizer.zero_grad(set_to_none=True); loss_total=torch.zeros((),device=device)
        for start in range(0,len(indices),batch):
            args=data.batch(indices[start:start+batch],device)
            loss=wrapper(*args).sum(); loss_total+=loss.detach()
            scaler.scale(loss/denominator).backward()
        scaler.unscale_(optimizer)
        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg['clip'])
        if device.type=='cpu' and not torch.isfinite(norm): raise RuntimeError('Non-finite gradients')
        old=scaler.get_scale(); scaler.step(optimizer); scaler.update()
        if scaler.get_scale()>=old:
            value=float(loss_total)/denominator
            if not math.isfinite(value): raise RuntimeError('Non-finite loss')
            return value
    raise RuntimeError('Repeated FP16 overflow; checkpoint position unchanged')


## D. GPU memory and speed check

This compares microbatches while keeping the effective batch fixed. Each candidate runs two maximum-size updates, including optimizer-state allocation. The synthetic batch combines the maximum token length and maximum scored-character count from training and validation. These are temporary models; no training checkpoint is changed. A passing configuration needs at least 1.5 GiB estimated free memory on each GPU.

Initial training settings are starting values, not measured optima. Training uses a new optimizer and a new NER schedule. The Phase-1 optimizer is not restored.


In [17]:
def sync_devices():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)

def gpu_sweep(cfg):
    if not torch.cuda.is_available(): raise RuntimeError('Enable GPU acceleration')
    results=[]; saved=rng_state()
    nt=max(end-start for data in (TRAIN_DATA,VAL_DATA) for _,start,end,_,_ in data.rows)
    nc=max(b-a for data in (TRAIN_DATA,VAL_DATA) for _,_,_,a,b in data.rows)
    gpus=torch.cuda.device_count()
    class StressData:
        def __init__(self):
            self.rows=[('stress',0,nt,0,nc)]*cfg['effective_batch']
            self.docs={'stress':dict(targets=np.zeros(nc,dtype=np.int64))}
        def batch(self,indices,device):
            b=len(indices)
            return (torch.randint(4,MODEL_CONFIG['vocab_size'],(b,nt),device=device),
                    torch.randint(0,nt,(b,nc),device=device),
                    torch.randint(0,cfg['char_buckets'],(b,nc),device=device),
                    torch.randint(0,256,(b,nc),device=device),torch.zeros((b,nc),dtype=torch.long,device=device))
    try:
        for ckpt in (True,False):
            for micro in (1,2,4):
                if cfg['effective_batch']%(micro*gpus): continue
                candidate=dict(cfg,micro_batch=micro,activation_checkpointing=ckpt)
                model=optimizer=scaler=wrapper=None
                gc.collect(); torch.cuda.empty_cache()
                other=[]
                for i in range(gpus):
                    with torch.cuda.device(i): free,total=torch.cuda.mem_get_info()
                    other.append(max(0,total-free-torch.cuda.memory_reserved(i)))
                    torch.cuda.reset_peak_memory_stats(i)
                row=dict(micro_batch=micro,activation_checkpointing=ckpt,passed=False)
                try:
                    seed_all(cfg['seed']); model,optimizer,scaler,wrapper,device=setup_training(candidate)
                    stress=StressData()
                    for _ in range(2): update_batch(model,optimizer,scaler,wrapper,stress,list(range(cfg['effective_batch'])),candidate,device)
                    indices=list(range(min(cfg['effective_batch'],len(TRAIN_DATA))))
                    update_batch(model,optimizer,scaler,wrapper,TRAIN_DATA,indices,candidate,device)
                    sync_devices(); started=time.monotonic()
                    for _ in range(3): update_batch(model,optimizer,scaler,wrapper,TRAIN_DATA,indices,candidate,device)
                    sync_devices(); seconds=(time.monotonic()-started)/3
                    memory=[]
                    for i in range(gpus):
                        total=torch.cuda.get_device_properties(i).total_memory; reserved=torch.cuda.max_memory_reserved(i)
                        memory.append(dict(gpu=i,reserved_gib=reserved/2**30,headroom_gib=(total-reserved-other[i])/2**30))
                    row.update(seconds_per_update=seconds,memory=memory,passed=all(m['headroom_gib']>=1.5 for m in memory))
                except torch.cuda.OutOfMemoryError: row['error']='out of memory'
                finally:
                    del model,optimizer,scaler,wrapper
                    gc.collect(); torch.cuda.empty_cache()
                results.append(row); print(row,flush=True)
                if 'error' in row: break
    finally: restore_rng(saved)
    safe=[r for r in results if r['passed']]
    if not safe: raise RuntimeError('No configuration passed; reduce the effective batch/window and rerun preparation')
    best=min(safe,key=lambda r:r['seconds_per_update'])
    write_json(ASSET_DIR/f"hardware_{cfg['method']}.json",dict(results=results,selected=best))
    return best


## Shared training and evaluation


In [18]:
def validate_predictions(predictions,texts,labels):
    if set(predictions)!=set(texts): raise ValueError('Prediction IDs do not match')
    for key,spans in predictions.items():
        if not isinstance(spans,list): raise ValueError(f'Invalid prediction: {key}')
        seen=set()
        for item in spans:
            if len(item)!=3: raise ValueError('Expected start, end, label')
            a,b,label=item
            if type(a)!=int or type(b)!=int or not 0<=a<b<=len(texts[key]) or label not in labels:
                raise ValueError(f'Invalid span: {key}: {item}')
            if tuple(item) in seen: raise ValueError(f'Duplicate span: {key}')
            seen.add(tuple(item))

def span_metrics(gold,predictions,labels):
    if set(gold)!=set(predictions): raise ValueError('Evaluation ID mismatch')
    counts={label:[0,0,0] for label in labels}
    for key in gold:
        g={tuple(s[:3]) for s in gold[key]}; p={tuple(s[:3]) for s in predictions[key]}
        for item in g&p: counts[item[2]][0]+=1
        for item in p-g: counts[item[2]][1]+=1
        for item in g-p: counts[item[2]][2]+=1
    scores={}
    for label,(tp,fp,fn) in counts.items():
        precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.
        scores[label]=dict(precision=precision,recall=recall,f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,tp=tp,fp=fp,fn=fn)
    return dict(macro_f1=float(np.mean([r['f1'] for r in scores.values()])),per_label=scores)

def viterbi_bio(logp):
    length,k=logp.shape
    if length==0: return np.empty(0,dtype=np.int64)
    if k%2!=1 or not np.isfinite(logp).all(): raise ValueError('Invalid BIO emissions')
    previous=np.arange(k)[:,None]; following=np.arange(k)[None,:]
    allowed=(following==0)|(following%2==1)|((previous>0)&((previous-1)//2==(following-1)//2))
    transition=np.where(allowed,0.,-np.inf).astype(np.float32)
    score=logp[0].copy(); score[2::2]=-np.inf
    back=np.zeros((length,k),dtype=np.int16)
    for t in range(1,length):
        candidates=score[:,None]+transition
        back[t]=candidates.argmax(axis=0)
        score=candidates.max(axis=0)+logp[t]
    path=np.zeros(length,dtype=np.int64); path[-1]=score.argmax()
    for t in range(length-1,0,-1): path[t-1]=back[t,path[t]]
    return path

@torch.inference_mode()
def predict_windows(model,data,labels,device):
    was_training=model.training; model.eval()
    emissions={key:np.zeros((len(doc['mapping']),1+2*len(labels)),dtype=np.float32)
               for key,doc in data.docs.items()}
    try:
        for index,(key,_,_,a,b) in enumerate(data.rows):
            args=data.batch([index],device)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=='cuda'):
                logits=model(*args[:-1])
            if not torch.isfinite(logits).all(): raise RuntimeError('Non-finite NER logits')
            emissions[key][a:b]=F.log_softmax(logits[0,:b-a].float(),dim=-1).cpu().numpy()
        return {key:decode_tags(viterbi_bio(logp),labels) for key,logp in emissions.items()}
    finally: model.train(was_training)

def experiment_signature(cfg):
    paths=['Train/fold1-text.json','Train/fold1-gold.json','Train/fold2-text.json',
           'Train/fold2-gold.json','Train/labels.txt','Test/fold3-text.json']
    return dict(version='phase2-model1-three-methods-v1',config=cfg,model=MODEL_CONFIG,
        parent_sha256=digest_file(BASE_CHECKPOINT),tokenizer_sha256=digest_file(BASE_RUN/'tokenizer.json'),
        parent_exposure=BASE_EXPOSURE,labels=LABELS,split=split,
        data={name:digest_file(DATA_ROOT/name) for name in paths},
        evaluator_sha256=digest_file(EVALUATOR_PATH),gpus=torch.cuda.device_count())

def save_training(s):
    atomic_save(dict(model=s.model.state_dict(),optimizer=s.optimizer.state_dict(),
        scaler=s.scaler.state_dict(),rng=rng_state(),signature=s.signature,
        epoch=s.epoch,position=s.position,step=s.step,best=s.best),s.out/'latest.pt')
    write_json(s.out/'progress.json',dict(epoch=s.epoch,position=s.position,step=s.step,
        best_macro_f1=s.best,complete=s.epoch>=s.cfg['epochs']))

def log_training(s,**row):
    record=dict(step=s.step,epoch=s.epoch,**row)
    with (s.out/'metrics.jsonl').open('a',encoding='utf-8') as handle:
        handle.write(json.dumps(record)+'\n')
    print(record,flush=True)

def evaluate_session(s):
    saved=rng_state()
    try:
        pred=predict_windows(s.model,VAL_DATA,LABELS,s.device)
        texts={k:TEXTS[k] for k in split['val']}; truth={k:GOLD[k] for k in split['val']}
        validate_predictions(pred,texts,LABELS)
        scores=span_metrics(truth,pred,LABELS)
        log_training(s,validation_macro_f1=scores['macro_f1'])
        if scores['macro_f1']>s.best:
            s.best=scores['macro_f1']
            atomic_save(dict(model=s.model.state_dict(),signature=s.signature,step=s.step,
                epoch=s.epoch,metrics=scores),s.out/'best.pt')
            write_json(s.out/'best_validation_predictions.json',pred)
            write_json(s.out/'best_validation_metrics.json',scores)
        return scores
    finally: restore_rng(saved)

def open_session(cfg,cpu=False):
    if not cpu and not torch.cuda.is_available(): raise RuntimeError('Enable GPUs')
    signature=experiment_signature(cfg)
    if cfg['effective_batch']%(max(1,signature['gpus'])*cfg['micro_batch']): raise ValueError('Effective batch mismatch')
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    config_path=RUN_DIR/'config.json'
    if config_path.exists():
        if read_json(config_path)!=signature: raise ValueError('Run configuration changed; use a new RUN_DIR')
    elif any(RUN_DIR.iterdir()): raise ValueError('Nonempty output directory without config.json')
    write_json(config_path,signature); write_json(RUN_DIR/'split.json',split)
    shutil.copy2(BASE_RUN/'tokenizer.json',RUN_DIR/'tokenizer.json')
    shutil.copy2(EVALUATOR_PATH,RUN_DIR/'evaluate_phase2.py')
    shutil.copy2(DATA_ROOT/'Train/labels.txt',RUN_DIR/'labels.txt')
    write_json(RUN_DIR/'validation_truth.json',{k:GOLD[k] for k in split['val']})
    seed_all(cfg['seed'])
    model,optimizer,scaler,wrapper,device=setup_training(cfg,cpu)
    s=SimpleNamespace(model=model,optimizer=optimizer,scaler=scaler,wrapper=wrapper,device=device,
        out=RUN_DIR,cfg=cfg,signature=signature,epoch=0,position=0,step=0,best=-1.)
    path=RUN_DIR/'latest.pt'
    if path.exists():
        state=torch.load(path,map_location='cpu',weights_only=False)
        if state['signature']!=signature: raise ValueError('Checkpoint signature mismatch')
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer']); scaler.load_state_dict(state['scaler'])
        for name in ('epoch','position','step','best'): setattr(s,name,state[name])
        restore_rng(state['rng'])
    return s

def schedule_ratio(step,total,cfg):
    warm=max(1,round(total*cfg['warmup_fraction']))
    if step<warm: return (step+1)/warm
    progress=min(1.,(step-warm)/max(1,total-warm))
    return cfg['min_lr_ratio']+(1-cfg['min_lr_ratio'])*0.5*(1+math.cos(math.pi*progress))

def train_run(cfg,additional_steps=None,cpu=False):
    s=open_session(cfg,cpu); started=time.monotonic(); initial=s.step
    total=math.ceil(len(TRAIN_DATA)/cfg['effective_batch'])*cfg['epochs']
    try:
        if s.best<0: evaluate_session(s); save_training(s)
        while s.epoch<cfg['epochs']:
            if additional_steps is not None and s.step-initial>=additional_steps: break
            if time.monotonic()-started>=cfg['max_hours']*3600: break
            order=np.random.default_rng(cfg['seed']+s.epoch).permutation(len(TRAIN_DATA))
            end=min(len(order),s.position+cfg['effective_batch'])
            indices=order[s.position:end]
            ratio=schedule_ratio(s.step,total,cfg)
            for group in s.optimizer.param_groups: group['lr']=group['base_lr']*ratio
            loss=update_batch(s.model,s.optimizer,s.scaler,s.wrapper,TRAIN_DATA,indices,cfg,s.device)
            s.position=end; s.step+=1
            if s.position==len(order): s.epoch+=1; s.position=0; epoch_end=True
            else: epoch_end=False
            if s.step%cfg['log_every']==0 or s.step==initial+1:
                log_training(s,loss=loss,backbone_lr=s.optimizer.param_groups[0]['lr'],
                    elapsed_minutes=(time.monotonic()-started)/60)
            if s.step%cfg['eval_every']==0 or epoch_end: evaluate_session(s); save_training(s)
            elif s.step%cfg['save_every']==0: save_training(s)
        evaluate_session(s); save_training(s)
        report=dict(steps=s.step-initial,seconds=time.monotonic()-started,
            best_macro_f1=s.best,complete=s.epoch>=cfg['epochs'])
        write_json(s.out/'session_report.json',report); print(report)
        return report
    finally:
        del s
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [19]:
def load_ner_checkpoint(path,device):
    state=torch.load(path,map_location='cpu',weights_only=False)
    signature=state['signature']; cfg=signature['config']
    if signature['labels']!=LABELS: raise ValueError('Checkpoint label order mismatch')
    if signature['tokenizer_sha256']!=digest_file(RUN_DIR/'tokenizer.json'): raise ValueError('Checkpoint tokenizer mismatch')
    model=LegalNER(signature['model'],1+2*len(LABELS),cfg,state['model']['class_weights'])
    model.load_state_dict(state['model']); model.to(device).eval()
    return model,signature

def predict_set(checkpoint_path,texts,out_root,cpu=False):
    device=torch.device('cpu' if cpu else 'cuda')
    signature=dict(checkpoint=digest_file(checkpoint_path),
        texts=hashlib.sha256(json.dumps(texts,sort_keys=True,ensure_ascii=False).encode('utf-8')).hexdigest(),
        policy='model1-character-bio-viterbi-v1')
    tag=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:16]
    out=Path(out_root)/tag; out.mkdir(parents=True,exist_ok=True)
    if (out/'generation.json').exists() and read_json(out/'generation.json')!=signature: raise ValueError('Prediction signature mismatch')
    write_json(out/'generation.json',signature)
    predictions=read_json(out/'prediction.json') if (out/'prediction.json').exists() else {}
    if not set(predictions)<=set(texts): raise ValueError('Unknown saved document IDs')
    validate_predictions(predictions,{k:texts[k] for k in predictions},LABELS)
    model,trained=load_ner_checkpoint(checkpoint_path,device)
    try:
        for key in texts:
            if key in predictions: continue
            data=WindowData(texts,[key],TOKENIZER,trained['config'])
            verify_windows(data,texts)
            result=predict_windows(model,data,LABELS,device)
            validate_predictions(result,{key:texts[key]},LABELS)
            predictions.update(result); write_json(out/'prediction.json',predictions)
            if len(predictions)%10==0: print(f'{len(predictions)}/{len(texts)} documents',flush=True)
        validate_predictions(predictions,texts,LABELS)
        temp=out/'submission.tmp.zip'; final=out/'submission.zip'
        with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as archive: archive.write(out/'prediction.json','prediction.json')
        with zipfile.ZipFile(temp) as archive:
            assert archive.namelist()==['prediction.json'] and archive.testzip() is None
            validate_predictions(json.loads(archive.read('prediction.json')),texts,LABELS)
        os.replace(temp,final)
        return out
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [20]:
def activate(method):
    global RUN_DIR,CONFIG
    if method not in ('full','adapters','lorma'): raise ValueError(method)
    RUN_DIR=COLLECTION/method
    config_path=RUN_DIR/'config.json'
    CONFIG=dict(COMMON_CONFIG,method=method)
    if method!='full': CONFIG['backbone_lr']=1e-4
    if config_path.exists():
        CONFIG=read_json(config_path)['config']
        if CONFIG['method']!=method: raise ValueError('Method mismatch')
    for name in ('seed','validation_fraction','window','overlap','char_buckets','class_weight_cap'):
        if CONFIG[name]!=COMMON_CONFIG[name]: raise ValueError(f'Shared preparation setting changed: {name}')
    print('Method:',method,'Folder:',RUN_DIR)

def check_method():
    if not (RUN_DIR/'config.json').exists():
        selected=gpu_sweep(CONFIG)
        CONFIG['micro_batch']=selected['micro_batch']
        CONFIG['activation_checkpointing']=selected['activation_checkpointing']
    probe=make_model(CONFIG,torch.device('cpu'))
    report=dict(total=sum(p.numel() for p in probe.parameters()),trainable=sum(p.numel() for p in probe.parameters() if p.requires_grad))
    write_json(ASSET_DIR/f"parameters_{CONFIG['method']}.json",report)
    print(report); del probe; gc.collect()

def evaluate_method():
    selected=RUN_DIR/'best.pt'
    out=predict_set(selected,{k:TEXTS[k] for k in split['val']},RUN_DIR/'validation_predictions'/digest_file(selected)[:12])
    command=[sys.executable,str(EVALUATOR_PATH),'--truth',str(RUN_DIR/'validation_truth.json'),
        '--prediction',str(out/'prediction.json'),'--labels',str(DATA_ROOT/'Train/labels.txt')]
    result=subprocess.run(command,capture_output=True,text=True)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    (out/'prof_evaluation.txt').write_text(result.stdout+'\n'+result.stderr,encoding='utf-8')
    write_json(out/'evaluator_provenance.json',dict(script_sha256=digest_file(EVALUATOR_PATH),
        truth_sha256=digest_file(RUN_DIR/'validation_truth.json'),prediction_sha256=digest_file(out/'prediction.json'),returncode=result.returncode))
    result.check_returncode()
    scores=span_metrics({k:GOLD[k] for k in split['val']},read_json(out/'prediction.json'),LABELS)
    match=re.search(r'Strict Macro-F1:\s*([0-9.]+)',result.stdout)
    assert match and abs(float(match.group(1))-scores['macro_f1'])<1e-6
    return out

def submit_method():
    import base64
    selected=RUN_DIR/'best.pt'
    out=predict_set(selected,TEST_TEXTS,RUN_DIR/'test_predictions'/digest_file(selected)[:12])
    archive=out/'submission.zip'; print(archive)
    if archive.stat().st_size<25*2**20:
        payload=base64.b64encode(archive.read_bytes()).decode('ascii')
        name=f"phase2_model1_{CONFIG['method']}.zip"
        display(HTML(f'<a download="{name}" href="data:application/zip;base64,{payload}">Download {name}</a>'))
    return out


## Full fine-tuning

Activate the method, run the optional 30-update check, then train and evaluate its best checkpoint.

In [1]:
activate('full')
check_method()


Method: full Folder: /kaggle/working/phase2_model1_comparison/full
{'micro_batch': 1, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.47750644766667466, 'memory': [{'gpu': 0, 'reserved_gib': 3.240234375, 'headroom_gib': 11.21954345703125}, {'gpu': 1, 'reserved_gib': 0.98828125, 'headroom_gib': 13.47149658203125}]}
{'micro_batch': 2, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.2910241289999931, 'memory': [{'gpu': 0, 'reserved_gib': 2.75, 'headroom_gib': 11.59649658203125}, {'gpu': 1, 'reserved_gib': 1.05859375, 'headroom_gib': 13.28790283203125}]}
{'micro_batch': 1, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.3810046996666567, 'memory': [{'gpu': 0, 'reserved_gib': 3.37109375, 'headroom_gib': 10.97540283203125}, {'gpu': 1, 'reserved_gib': 1.1640625, 'headroom_gib': 13.18243408203125}]}
{'micro_batch': 2, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.26694895000000923, 'memo

In [2]:
short_report=train_run(CONFIG,additional_steps=30)


{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.0}
{'step': 1, 'epoch': 0, 'loss': 3.2434754425319223, 'backbone_lr': 1.567398119122257e-07, 'elapsed_minutes': 0.11052511733333385}
{'step': 20, 'epoch': 0, 'loss': 2.612770343963319, 'backbone_lr': 3.134796238244514e-06, 'elapsed_minutes': 0.19622834411666665}
{'step': 30, 'epoch': 0, 'validation_macro_f1': 0.0}
{'steps': 30, 'seconds': 20.845321219000027, 'best_macro_f1': 0.0, 'complete': False}


In [3]:
training_report=train_run(CONFIG)


{'step': 31, 'epoch': 0, 'loss': 1.7932531926172042, 'backbone_lr': 4.8589341692789965e-06, 'elapsed_minutes': 0.005558280583333423}
{'step': 40, 'epoch': 0, 'loss': 1.249479604041021, 'backbone_lr': 6.269592476489028e-06, 'elapsed_minutes': 0.04629496399999956}
{'step': 60, 'epoch': 0, 'loss': 0.996488258710226, 'backbone_lr': 9.404388714733543e-06, 'elapsed_minutes': 0.13719223149999968}
{'step': 80, 'epoch': 0, 'loss': 1.070418329244673, 'backbone_lr': 1.2539184952978056e-05, 'elapsed_minutes': 0.2284776120833328}
{'step': 100, 'epoch': 0, 'loss': 0.5370050119637737, 'backbone_lr': 1.5673981191222574e-05, 'elapsed_minutes': 0.3242628681833329}
{'step': 120, 'epoch': 0, 'loss': 0.59876894491534, 'backbone_lr': 1.8808777429467086e-05, 'elapsed_minutes': 0.4454649276999997}
{'step': 133, 'epoch': 1, 'validation_macro_f1': 0.11193033674404347}
{'step': 140, 'epoch': 1, 'loss': 0.36036975554179596, 'backbone_lr': 2.19435736677116e-05, 'elapsed_minutes': 0.6677785616999993}
{'step': 160, 

In [4]:
validation_output=evaluate_method()


10/10 documents
Examples: 10
APP: P=0.073529 R=0.090909 F1=0.081301
RESP: P=0.109375 R=0.250000 F1=0.152174
A.COUNSEL: P=0.551724 R=0.301887 F1=0.390244
R.COUNSEL: P=0.373134 R=0.806452 F1=0.510204
JUDGE: P=0.589744 R=0.793103 F1=0.676471
WIT: P=0.279070 R=0.800000 F1=0.413793
AUTH: P=0.258242 R=0.470000 F1=0.333333
COURT: P=0.657895 R=0.847458 F1=0.740741
STAT: P=0.500000 R=0.708738 F1=0.586345
PREC: P=0.270833 R=0.414013 F1=0.327456
DATE: P=0.837302 R=0.905579 F1=0.870103
CASENO: P=0.397849 R=0.528571 F1=0.453988
Strict Macro-F1:        0.461346
Strict Macro-Precision: 0.408225
Strict Macro-Recall:    0.576393



In [5]:
test_output=submit_method()


10/35 documents
20/35 documents
30/35 documents
/kaggle/working/phase2_model1_comparison/full/test_predictions/612e2b2a77ca/938ff409d7cfcfa9/submission.zip


## Bottleneck adapters

Activate the method, run the optional 30-update check, then train and evaluate its best checkpoint.

In [6]:
activate('adapters')
check_method()


Method: adapters Folder: /kaggle/working/phase2_model1_comparison/adapters
{'micro_batch': 1, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.4407150053333074, 'memory': [{'gpu': 0, 'reserved_gib': 1.787109375, 'headroom_gib': 12.17071533203125}, {'gpu': 1, 'reserved_gib': 0.638671875, 'headroom_gib': 13.70782470703125}]}
{'micro_batch': 2, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.2719235253333257, 'memory': [{'gpu': 0, 'reserved_gib': 1.76171875, 'headroom_gib': 12.19610595703125}, {'gpu': 1, 'reserved_gib': 0.767578125, 'headroom_gib': 13.57891845703125}]}
{'micro_batch': 1, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.3325678370000181, 'memory': [{'gpu': 0, 'reserved_gib': 1.810546875, 'headroom_gib': 12.14727783203125}, {'gpu': 1, 'reserved_gib': 1.091796875, 'headroom_gib': 13.25469970703125}]}
{'micro_batch': 2, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.221819

In [7]:
short_report=train_run(CONFIG,additional_steps=30)


{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.0}
{'step': 1, 'epoch': 0, 'loss': 3.2434754425319223, 'backbone_lr': 3.134796238244514e-07, 'elapsed_minutes': 0.11180995975000012}
{'step': 20, 'epoch': 0, 'loss': 2.711577052990392, 'backbone_lr': 6.269592476489028e-06, 'elapsed_minutes': 0.18252294851667253}
{'step': 30, 'epoch': 0, 'validation_macro_f1': 0.0}
{'steps': 30, 'seconds': 18.66934781000009, 'best_macro_f1': 0.0, 'complete': False}


In [8]:
training_report=train_run(CONFIG)


{'step': 31, 'epoch': 0, 'loss': 1.997552352657135, 'backbone_lr': 9.717868338557993e-06, 'elapsed_minutes': 0.0044938444833330015}
{'step': 40, 'epoch': 0, 'loss': 1.3493836679644757, 'backbone_lr': 1.2539184952978056e-05, 'elapsed_minutes': 0.03807542141666242}
{'step': 60, 'epoch': 0, 'loss': 1.0638444960598505, 'backbone_lr': 1.8808777429467086e-05, 'elapsed_minutes': 0.113011089833329}
{'step': 80, 'epoch': 0, 'loss': 1.277506686090961, 'backbone_lr': 2.5078369905956112e-05, 'elapsed_minutes': 0.18874440738333456}
{'step': 100, 'epoch': 0, 'loss': 0.7264826223745355, 'backbone_lr': 3.134796238244515e-05, 'elapsed_minutes': 0.2655793238666623}
{'step': 120, 'epoch': 0, 'loss': 0.766171905305946, 'backbone_lr': 3.761755485893417e-05, 'elapsed_minutes': 0.3535023656333351}
{'step': 133, 'epoch': 1, 'validation_macro_f1': 0.032527458758375045}
{'step': 140, 'epoch': 1, 'loss': 0.4844546558747502, 'backbone_lr': 4.38871473354232e-05, 'elapsed_minutes': 0.5305626065333323}
{'step': 160,

In [9]:
validation_output=evaluate_method()


10/10 documents
Examples: 10
APP: P=0.224490 R=0.200000 F1=0.211538
RESP: P=0.136364 R=0.214286 F1=0.166667
A.COUNSEL: P=0.424242 R=0.264151 F1=0.325581
R.COUNSEL: P=0.384615 R=0.645161 F1=0.481928
JUDGE: P=0.552632 R=0.724138 F1=0.626866
WIT: P=0.188679 R=0.666667 F1=0.294118
AUTH: P=0.307692 R=0.440000 F1=0.362140
COURT: P=0.679487 R=0.898305 F1=0.773723
STAT: P=0.507937 R=0.621359 F1=0.558952
PREC: P=0.285714 R=0.292994 F1=0.289308
DATE: P=0.884774 R=0.922747 F1=0.903361
CASENO: P=0.475610 R=0.557143 F1=0.513158
Strict Macro-F1:        0.458945
Strict Macro-Precision: 0.421020
Strict Macro-Recall:    0.537246



In [10]:
test_output=submit_method()


10/35 documents
20/35 documents
30/35 documents
/kaggle/working/phase2_model1_comparison/adapters/test_predictions/0468c9cd91de/0f074c9bba8629f4/submission.zip


## LoRMA+ multiplicative adaptation

Activate the method, run the optional 30-update check, then train and evaluate its best checkpoint.

In [11]:
activate('lorma')
check_method()


Method: lorma Folder: /kaggle/working/phase2_model1_comparison/lorma
{'micro_batch': 1, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.5360192543333445, 'memory': [{'gpu': 0, 'reserved_gib': 1.90234375, 'headroom_gib': 12.05548095703125}, {'gpu': 1, 'reserved_gib': 0.638671875, 'headroom_gib': 13.70782470703125}]}
{'micro_batch': 2, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 0.2800106263333267, 'memory': [{'gpu': 0, 'reserved_gib': 1.716796875, 'headroom_gib': 12.24102783203125}, {'gpu': 1, 'reserved_gib': 0.74609375, 'headroom_gib': 13.60040283203125}]}
{'micro_batch': 1, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.400362570000046, 'memory': [{'gpu': 0, 'reserved_gib': 1.904296875, 'headroom_gib': 12.05157470703125}, {'gpu': 1, 'reserved_gib': 1.12890625, 'headroom_gib': 13.21759033203125}]}
{'micro_batch': 2, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 0.242894332666764

In [12]:
short_report=train_run(CONFIG,additional_steps=30)


{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.0}
{'step': 1, 'epoch': 0, 'loss': 3.2434849554716387, 'backbone_lr': 3.134796238244514e-07, 'elapsed_minutes': 0.11700768613333518}
{'step': 20, 'epoch': 0, 'loss': 2.7314880679178994, 'backbone_lr': 6.269592476489028e-06, 'elapsed_minutes': 0.1918979917666699}
{'step': 30, 'epoch': 0, 'validation_macro_f1': 0.0}
{'steps': 30, 'seconds': 19.73680639000031, 'best_macro_f1': 0.0, 'complete': False}


In [13]:
training_report=train_run(CONFIG)


{'step': 31, 'epoch': 0, 'loss': 2.0415726675861916, 'backbone_lr': 9.717868338557993e-06, 'elapsed_minutes': 0.004808421366654632}
{'step': 40, 'epoch': 0, 'loss': 1.3782456676082144, 'backbone_lr': 1.2539184952978056e-05, 'elapsed_minutes': 0.0402908595666607}
{'step': 60, 'epoch': 0, 'loss': 1.0634753686079983, 'backbone_lr': 1.8808777429467086e-05, 'elapsed_minutes': 0.11921029078333352}
{'step': 80, 'epoch': 0, 'loss': 1.2893338674076165, 'backbone_lr': 2.5078369905956112e-05, 'elapsed_minutes': 0.1983883115499945}
{'step': 100, 'epoch': 0, 'loss': 0.7569469674427547, 'backbone_lr': 3.134796238244515e-05, 'elapsed_minutes': 0.2782555946166667}
{'step': 120, 'epoch': 0, 'loss': 0.8273547067995062, 'backbone_lr': 3.761755485893417e-05, 'elapsed_minutes': 0.3690357951499967}
{'step': 133, 'epoch': 1, 'validation_macro_f1': 0.013487073012976435}
{'step': 140, 'epoch': 1, 'loss': 0.5296053650210057, 'backbone_lr': 4.38871473354232e-05, 'elapsed_minutes': 0.5546119077666541}
{'step': 16

In [14]:
validation_output=evaluate_method()


10/10 documents
Examples: 10
APP: P=0.093023 R=0.072727 F1=0.081633
RESP: P=0.132353 R=0.321429 F1=0.187500
A.COUNSEL: P=0.333333 R=0.264151 F1=0.294737
R.COUNSEL: P=0.377358 R=0.645161 F1=0.476190
JUDGE: P=0.461538 R=0.620690 F1=0.529412
WIT: P=0.281250 R=0.600000 F1=0.382979
AUTH: P=0.243523 R=0.470000 F1=0.320819
COURT: P=0.741935 R=0.779661 F1=0.760331
STAT: P=0.428571 R=0.553398 F1=0.483051
PREC: P=0.353933 R=0.401274 F1=0.376119
DATE: P=0.902954 R=0.918455 F1=0.910638
CASENO: P=0.441860 R=0.542857 F1=0.487179
Strict Macro-F1:        0.440882
Strict Macro-Precision: 0.399303
Strict Macro-Recall:    0.515817



In [15]:
test_output=submit_method()


10/35 documents
20/35 documents
30/35 documents
/kaggle/working/phase2_model1_comparison/lorma/test_predictions/840b0a82420d/c544546a9e50332c/submission.zip


## Bottleneck adapters with a character-level CRF

Run the adapter section first or restore its configuration and validation artifacts. The CRF uses the same character windows and label order. The saved run continued through epochs 10, 20, and 35. Test inference uses the best validation checkpoint.

In [18]:
from contextlib import contextmanager
import itertools
import base64

CRF_ROOT = Path("/kaggle/working/model1_adapters_crf_v1")
CRF_RUN = CRF_ROOT / "run"
CRF_CHECKS = CRF_ROOT / "checks"
CRF_BASELINE = Path(COLLECTION) / "adapters"

crf_baseline_signature = read_json(CRF_BASELINE / "config.json")

assert crf_baseline_signature["config"]["method"] == "adapters"
assert crf_baseline_signature["split"] == split, "Validation split changed"
assert crf_baseline_signature["labels"] == LABELS, "Label order changed"
assert crf_baseline_signature["model"] == MODEL_CONFIG, "Architecture changed"
assert crf_baseline_signature["parent_sha256"] == digest_file(BASE_CHECKPOINT), (
    "Use the same Phase-1 checkpoint as the original adapters"
)
assert crf_baseline_signature["tokenizer_sha256"] == digest_file(
    BASE_RUN / "tokenizer.json"
), "Tokenizer changed"

for filename, expected_hash in crf_baseline_signature["data"].items():
    assert digest_file(DATA_ROOT / filename) == expected_hash, filename

for name in ("window", "overlap", "char_buckets"):
    assert TRAIN_DATA.cfg[name] == crf_baseline_signature["config"][name]
    assert VAL_DATA.cfg[name] == crf_baseline_signature["config"][name]

CRF_CONFIG = copy.deepcopy(crf_baseline_signature["config"])
CRF_CONFIG.update(
    head_version="model1-adapters-crf-v1",
    crf_version=1,
    loss="unweighted-crf-normalized-by-valid-characters",
)

if (CRF_RUN / "config.json").exists():
    saved_cfg = read_json(CRF_RUN / "config.json")["config"]
    hardware_keys = {"micro_batch", "activation_checkpointing"}
    expected = {k: v for k, v in CRF_CONFIG.items() if k not in hardware_keys}
    actual = {k: v for k, v in saved_cfg.items() if k not in hardware_keys}
    assert expected == actual, "CRF configuration changed; choose a new CRF_ROOT"
    CRF_CONFIG = saved_cfg

CRF_CHECKS.mkdir(parents=True, exist_ok=True)

# Keep the original class available when this cell is rerun.
CRF_BASE_NER = LegalNER

print("Baseline:", CRF_BASELINE)
print("New experiment:", CRF_ROOT)
print("Phase-1 checkpoint:", BASE_CHECKPOINT)
print("Epochs:", CRF_CONFIG["epochs"])

Baseline: /kaggle/working/phase2_model1_comparison/adapters
New experiment: /kaggle/working/model1_adapters_crf_v1
Phase-1 checkpoint: /kaggle/input/datasets/acadsiitk/model1-phase2-data/best.pt
Epochs: 40


In [20]:
def crf_allowed_transitions(k):
    previous = np.arange(k)[:, None]
    following = np.arange(k)[None, :]
    return (
        (following == 0)
        | (following % 2 == 1)
        | (
            (previous > 0)
            & ((previous - 1) // 2 == (following - 1) // 2)
        )
    )


def crf_lse(values, axis):
    maximum = np.max(values, axis=axis, keepdims=True)
    result = maximum + np.log(
        np.exp(values - maximum).sum(axis=axis, keepdims=True)
    )
    return np.squeeze(result, axis=axis)


def crf_loss_and_gradients(emissions, transitions, tags):
    length, k = emissions.shape
    if length == 0:
        raise ValueError("Empty CRF training sequence")
    if not np.isfinite(emissions).all() or not np.isfinite(transitions).all():
        raise RuntimeError("Non-finite CRF inputs")

    allowed = crf_allowed_transitions(k)
    if np.any(tags < 0) or np.any(tags >= k):
        raise ValueError("Invalid CRF labels")
    if length > 1 and not allowed[tags[:-1], tags[1:]].all():
        raise ValueError("Invalid internal BIO transition in gold labels")

    tr = np.where(allowed, transitions, -np.inf)
    alpha = np.empty_like(emissions)
    normalizers = np.empty(length, dtype=np.float64)

    # All initial labels are allowed in a cropped training region.
    normalizers[0] = crf_lse(emissions[0], 0)
    alpha[0] = emissions[0] - normalizers[0]

    for t in range(1, length):
        scores = emissions[t] + crf_lse(
            alpha[t - 1, :, None] + tr, 0
        )
        normalizers[t] = crf_lse(scores, 0)
        alpha[t] = scores - normalizers[t]

    gold_score = emissions[np.arange(length), tags].sum()
    if length > 1:
        gold_score += tr[tags[:-1], tags[1:]].sum()

    loss = normalizers.sum() - gold_score
    emission_grad = np.empty_like(emissions)
    transition_grad = np.zeros_like(transitions)

    beta = np.zeros(k, dtype=np.float64)
    emission_grad[-1] = np.exp(alpha[-1])

    for t in range(length - 1, 0, -1):
        suffix = (
            tr
            + emissions[t][None, :]
            + beta[None, :]
            - normalizers[t]
        )
        transition_grad += np.exp(alpha[t - 1, :, None] + suffix)
        beta = crf_lse(suffix, 1)
        emission_grad[t - 1] = np.exp(alpha[t - 1] + beta)

    emission_grad[np.arange(length), tags] -= 1
    if length > 1:
        np.add.at(transition_grad, (tags[:-1], tags[1:]), -1)
    transition_grad[~allowed] = 0

    return float(loss), emission_grad, transition_grad


class CRFLossFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, emissions, transitions, targets):
        e = emissions.detach().double().cpu().numpy()
        tr = transitions.detach().double().cpu().numpy()
        y = targets.detach().cpu().numpy()

        grad_e = np.zeros_like(e)
        grad_tr = np.zeros_like(tr)
        total_loss = 0.0

        for row in range(len(e)):
            valid = y[row] != -100
            length = int(valid.sum())

            if length == 0 or not np.array_equal(
                valid, np.arange(len(valid)) < length
            ):
                raise ValueError("CRF requires nonempty, right-padded sequences")

            loss, de, dt = crf_loss_and_gradients(
                e[row, :length], tr, y[row, :length]
            )
            total_loss += loss
            grad_e[row, :length] = de
            grad_tr += dt

        ctx.save_for_backward(
            torch.as_tensor(
                grad_e, device=emissions.device, dtype=emissions.dtype
            ),
            torch.as_tensor(
                grad_tr, device=transitions.device, dtype=transitions.dtype
            ),
        )
        return emissions.new_tensor(total_loss)

    @staticmethod
    def backward(ctx, upstream_gradient):
        grad_e, grad_tr = ctx.saved_tensors
        return (
            grad_e * upstream_gradient,
            grad_tr * upstream_gradient,
            None,
        )


class AdapterCRFNER(CRF_BASE_NER):
    def __init__(self, model_cfg, nlabels, cfg, weights):
        super().__init__(model_cfg, nlabels, cfg, weights)
        self.crf_transitions = nn.Parameter(
            torch.zeros(nlabels, nlabels)
        )

    def forward(self, x, mapping, chars, positions, y=None):
        logits = super().forward(x, mapping, chars, positions)

        if y is None:
            return logits

        if not torch.equal(mapping.ge(0), y.ne(-100)):
            raise ValueError("Character padding mask mismatch")

        return CRFLossFunction.apply(
            logits.float(), self.crf_transitions.float(), y
        ).reshape(1)


def crf_decode(emissions, transitions, document_start=True):
    length, k = emissions.shape
    if length == 0:
        return np.empty(0, dtype=np.int64)
    if not np.isfinite(emissions).all() or not np.isfinite(transitions).all():
        raise RuntimeError("Non-finite CRF decoding inputs")

    tr = np.where(crf_allowed_transitions(k), transitions, -np.inf)
    score = emissions[0].astype(np.float64).copy()

    if document_start:
        score[2::2] = -np.inf

    back = np.zeros((length, k), dtype=np.int16)

    for t in range(1, length):
        candidates = score[:, None] + tr
        back[t] = candidates.argmax(axis=0)
        score = candidates.max(axis=0) + emissions[t]
        score -= score.max()

    path = np.empty(length, dtype=np.int64)
    path[-1] = score.argmax()
    for t in range(length - 1, 0, -1):
        path[t - 1] = back[t, path[t]]

    return path


@torch.inference_mode()
def crf_predict_windows(model, data, labels, device):
    was_training = model.training
    model.eval()

    emissions = {
        key: np.zeros(
            (len(doc["mapping"]), 1 + 2 * len(labels)),
            dtype=np.float32,
        )
        for key, doc in data.docs.items()
    }
    transitions = model.crf_transitions.detach().float().cpu().numpy()

    try:
        for index, (key, _, _, a, b) in enumerate(data.rows):
            args = data.batch([index], device)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=device.type == "cuda",
            ):
                logits = model(*args[:-1])

            if not torch.isfinite(logits).all():
                raise RuntimeError("Non-finite NER logits")

            emissions[key][a:b] = (
                logits[0, :b-a].float().cpu().numpy()
            )

        return {
            key: decode_tags(crf_decode(values, transitions), labels)
            for key, values in emissions.items()
        }
    finally:
        model.train(was_training)


@contextmanager
def crf_experiment():
    replacements = {
        "RUN_DIR": CRF_RUN,
        "ASSET_DIR": CRF_CHECKS,
        "CONFIG": CRF_CONFIG,
        "LegalNER": AdapterCRFNER,
        "predict_windows": crf_predict_windows,
        # The existing training loop uses these weights for its denominator.
        # Ones give normalization by the number of valid characters.
        "CLASS_WEIGHTS": torch.ones(1 + 2 * len(LABELS)),
    }
    previous = {name: globals()[name] for name in replacements}
    try:
        globals().update(replacements)
        yield
    finally:
        globals().update(previous)

In [21]:
def check_crf_implementation():
    saved_rng = rng_state()
    try:
        torch.manual_seed(71)
        k = 5
        allowed = crf_allowed_transitions(k)

        # Includes a window beginning inside an entity and a one-character window.
        for gold_tags in ([1, 2, 0], [2, 2, 0], [4]):
            length = len(gold_tags)
            e = torch.randn(
                1, length, k, dtype=torch.float64, requires_grad=True
            )
            tr = torch.randn(
                k, k, dtype=torch.float64, requires_grad=True
            )
            targets = torch.tensor([gold_tags])

            paths = [
                p for p in itertools.product(range(k), repeat=length)
                if all(
                    allowed[p[t-1], p[t]]
                    for t in range(1, length)
                )
            ]
            path_scores = torch.stack([
                sum(e[0, t, p[t]] for t in range(length))
                + sum(tr[p[t-1], p[t]] for t in range(1, length))
                for p in paths
            ])
            gold_score = (
                sum(e[0, t, gold_tags[t]] for t in range(length))
                + sum(
                    tr[gold_tags[t-1], gold_tags[t]]
                    for t in range(1, length)
                )
            )
            reference = (
                torch.logsumexp(path_scores, 0)
                - gold_score
                + tr.sum() * 0
            )
            actual = CRFLossFunction.apply(e, tr, targets)

            torch.testing.assert_close(
                actual, reference, atol=1e-9, rtol=1e-9
            )
            actual_grads = torch.autograd.grad(
                actual, (e, tr), retain_graph=True
            )
            reference_grads = torch.autograd.grad(reference, (e, tr))

            for actual_grad, reference_grad in zip(
                actual_grads, reference_grads
            ):
                torch.testing.assert_close(
                    actual_grad, reference_grad,
                    atol=1e-9, rtol=1e-9,
                )

            decoded = crf_decode(
                e.detach().numpy()[0],
                tr.detach().numpy(),
                document_start=False,
            )
            assert tuple(decoded) == paths[int(path_scores.argmax())]

        # Padding must contribute neither loss nor emission gradients.
        e = torch.randn(
            2, 4, k, dtype=torch.float64, requires_grad=True
        )
        tr = torch.zeros(k, k, dtype=torch.float64, requires_grad=True)
        y = torch.tensor([[1, 2, -100, -100], [0, 3, 4, 0]])

        batched = CRFLossFunction.apply(e, tr, y)
        separate = (
            CRFLossFunction.apply(e[:1, :2], tr, y[:1, :2])
            + CRFLossFunction.apply(e[1:], tr, y[1:])
        )
        torch.testing.assert_close(batched, separate)
        gradient = torch.autograd.grad(batched, e)[0]
        assert torch.count_nonzero(gradient[0, 2:]) == 0

        print("PASS: CRF loss, gradients, decoding, padding and window starts.")
    finally:
        restore_rng(saved_rng)


check_crf_implementation()

with crf_experiment():
    if not (CRF_RUN / "config.json").exists():
        selected = gpu_sweep(CRF_CONFIG)
        CRF_CONFIG["micro_batch"] = selected["micro_batch"]
        CRF_CONFIG["activation_checkpointing"] = (
            selected["activation_checkpointing"]
        )

        updates_per_epoch = math.ceil(
            len(TRAIN_DATA) / CRF_CONFIG["effective_batch"]
        )
        estimated_minutes = (
            updates_per_epoch * selected["seconds_per_update"] / 60
        )
        print("Training-only minutes per epoch:", round(estimated_minutes, 2))
    else:
        print("Resuming with saved CRF batch settings.")

write_json(CRF_CHECKS / "selected_config.json", CRF_CONFIG)
print("Configuration:", CRF_CONFIG)

PASS: CRF loss, gradients, decoding, padding and window starts.
{'micro_batch': 1, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 3.465631738000108, 'memory': [{'gpu': 0, 'reserved_gib': 1.814453125, 'headroom_gib': 12.14141845703125}, {'gpu': 1, 'reserved_gib': 0.640625, 'headroom_gib': 13.70587158203125}]}
{'micro_batch': 2, 'activation_checkpointing': True, 'passed': True, 'seconds_per_update': 3.3880870200000572, 'memory': [{'gpu': 0, 'reserved_gib': 1.75390625, 'headroom_gib': 12.20196533203125}, {'gpu': 1, 'reserved_gib': 0.751953125, 'headroom_gib': 13.59454345703125}]}
{'micro_batch': 1, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 3.457743235666688, 'memory': [{'gpu': 0, 'reserved_gib': 1.814453125, 'headroom_gib': 12.14141845703125}, {'gpu': 1, 'reserved_gib': 1.09375, 'headroom_gib': 13.25274658203125}]}
{'micro_batch': 2, 'activation_checkpointing': False, 'passed': True, 'seconds_per_update': 3.1404787760002364, 'memory'

In [ ]:
with crf_experiment():
    if not (CRF_RUN / "latest.pt").exists():
        crf_short_report = train_run(CRF_CONFIG, additional_steps=30)

### Train through epoch 10

The saved log begins after an interrupted session. A fresh run starts this stage after the 30-update check.

In [25]:
with crf_experiment():
    progress = read_json(CRF_RUN / "progress.json")
    updates_per_epoch = math.ceil(
        len(TRAIN_DATA) / CRF_CONFIG["effective_batch"]
    )
    remaining_updates = max(
        0, 10 * updates_per_epoch - progress["step"]
    )

    if remaining_updates:
        report = train_run(
            CRF_CONFIG,
            additional_steps=remaining_updates,
        )
        print(report, flush=True)
    else:
        print("Already reached epoch 10.")

{'step': 701, 'epoch': 5, 'loss': 0.008333832013039203, 'backbone_lr': 9.871724320040506e-05, 'elapsed_minutes': 0.04483132106667351}
{'step': 720, 'epoch': 5, 'loss': 0.012090972971772378, 'backbone_lr': 9.858680460653736e-05, 'elapsed_minutes': 0.927287361233357}
{'step': 740, 'epoch': 5, 'loss': 0.02224947810412123, 'backbone_lr': 9.84427930732043e-05, 'elapsed_minutes': 1.8398633712833543}
{'step': 760, 'epoch': 5, 'loss': 0.007334229927463657, 'backbone_lr': 9.829192416242933e-05, 'elapsed_minutes': 2.8042673113166834}
{'step': 780, 'epoch': 5, 'loss': 0.011923384363030079, 'backbone_lr': 9.813422168863576e-05, 'elapsed_minutes': 3.7882757545999994}
{'step': 798, 'epoch': 6, 'validation_macro_f1': 0.45449755162594835}
{'step': 800, 'epoch': 6, 'loss': 0.00595816086403466, 'backbone_lr': 9.79697105449143e-05, 'elapsed_minutes': 4.850377545899998}
{'step': 820, 'epoch': 6, 'loss': 0.004681458487383084, 'backbone_lr': 9.779841669909354e-05, 'elapsed_minutes': 5.859750311050023}
{'ste

### Train through epoch 20

In [26]:
with crf_experiment():
    progress = read_json(CRF_RUN / "progress.json")
    updates_per_epoch = math.ceil(
        len(TRAIN_DATA) / CRF_CONFIG["effective_batch"]
    )
    remaining_updates = max(
        0, 20 * updates_per_epoch - progress["step"]
    )

    if remaining_updates:
        report = train_run(
            CRF_CONFIG,
            additional_steps=remaining_updates,
        )
        print(report, flush=True)
    else:
        print("Already reached epoch 20.")

{'step': 1331, 'epoch': 10, 'loss': 0.0019108798052813556, 'backbone_lr': 9.122547639885794e-05, 'elapsed_minutes': 0.04343929836668394}
{'step': 1340, 'epoch': 10, 'loss': 0.0026854438304965205, 'backbone_lr': 9.107396202787484e-05, 'elapsed_minutes': 0.47431296796667083}
{'step': 1360, 'epoch': 10, 'loss': 0.0031978165403041313, 'backbone_lr': 9.073314219465941e-05, 'elapsed_minutes': 1.4198658455999975}
{'step': 1380, 'epoch': 10, 'loss': 0.0029356960198418185, 'backbone_lr': 9.038668194042404e-05, 'elapsed_minutes': 2.386412360433345}
{'step': 1400, 'epoch': 10, 'loss': 0.003529592583059528, 'backbone_lr': 9.003463595338197e-05, 'elapsed_minutes': 3.3359238558833266}
{'step': 1420, 'epoch': 10, 'loss': 0.001914748473269592, 'backbone_lr': 8.967705980344577e-05, 'elapsed_minutes': 4.334249466233329}
{'step': 1440, 'epoch': 10, 'loss': 0.0016131661544270752, 'backbone_lr': 8.931400993345551e-05, 'elapsed_minutes': 5.28105628785}
{'step': 1460, 'epoch': 10, 'loss': 0.00583357583725616

### Train through epoch 35

In [27]:
with crf_experiment():
    progress = read_json(CRF_RUN / "progress.json")
    updates_per_epoch = math.ceil(
        len(TRAIN_DATA) / CRF_CONFIG["effective_batch"]
    )
    remaining = max(0, 35 * updates_per_epoch - progress["step"])

    if remaining:
        report = train_run(CRF_CONFIG, additional_steps=remaining)
        print(report, flush=True)
    else:
        print("Already reached epoch 35.")

{'step': 2661, 'epoch': 20, 'loss': 0.0003206301729638589, 'backbone_lr': 5.9501313909878544e-05, 'elapsed_minutes': 0.057760275849977914}
{'step': 2680, 'epoch': 20, 'loss': 0.0013717637957919603, 'backbone_lr': 5.896659489929179e-05, 'elapsed_minutes': 0.9992634980166258}
{'step': 2700, 'epoch': 20, 'loss': 0.0004024554263673479, 'backbone_lr': 5.840312375923448e-05, 'elapsed_minutes': 1.98591221321664}
{'step': 2720, 'epoch': 20, 'loss': 0.0002805393321457451, 'backbone_lr': 5.7839115441375067e-05, 'elapsed_minutes': 2.9958191840999766}
{'step': 2740, 'epoch': 20, 'loss': 0.00028276424092566556, 'backbone_lr': 5.727465897355095e-05, 'elapsed_minutes': 4.037573956183284}
{'step': 2760, 'epoch': 20, 'loss': 0.00019630273081429025, 'backbone_lr': 5.6709843454339265e-05, 'elapsed_minutes': 5.035950946633299}
{'step': 2780, 'epoch': 20, 'loss': 0.0002767284603861115, 'backbone_lr': 5.6144758038992826e-05, 'elapsed_minutes': 6.014586437966622}
{'step': 2793, 'epoch': 21, 'validation_macro

### Validation comparison and test submission

In [28]:
with crf_experiment():
    CRF_DEV_OUT = evaluate_method()

crf_validation_scores = span_metrics(
    {key: GOLD[key] for key in split["val"]},
    read_json(CRF_DEV_OUT / "prediction.json"),
    LABELS,
)
crf_baseline_scores = read_json(
    CRF_BASELINE / "best_validation_metrics.json"
)

print("\nValidation comparison")
print("Original adapters:", crf_baseline_scores["macro_f1"])
print("Adapters + CRF:   ", crf_validation_scores["macro_f1"])
print(
    "Difference:       ",
    crf_validation_scores["macro_f1"] - crf_baseline_scores["macro_f1"],
)

for label in LABELS:
    before = crf_baseline_scores["per_label"][label]["f1"]
    after = crf_validation_scores["per_label"][label]["f1"]
    print(f"{label:12} {before:.4f} -> {after:.4f}")

write_json(
    CRF_RUN / "baseline_comparison.json",
    dict(
        baseline=crf_baseline_scores,
        adapters_crf=crf_validation_scores,
    ),
)

with crf_experiment():
    CRF_TEST_OUT = predict_set(
        CRF_RUN / "best.pt",
        TEST_TEXTS,
        CRF_RUN / "test_predictions",
    )

crf_submission = CRF_TEST_OUT / "submission.zip"
print("\nSubmission:", crf_submission)

crf_payload = base64.b64encode(crf_submission.read_bytes()).decode("ascii")
display(HTML(
    '<a download="model1_adapters_crf_submission.zip" '
    f'href="data:application/zip;base64,{crf_payload}">'
    'Download adapter + CRF submission</a>'
))

10/10 documents
Examples: 10
APP: P=0.531250 R=0.309091 F1=0.390805
RESP: P=0.258065 R=0.285714 F1=0.271186
A.COUNSEL: P=0.392857 R=0.207547 F1=0.271605
R.COUNSEL: P=0.392157 R=0.645161 F1=0.487805
JUDGE: P=0.636364 R=0.724138 F1=0.677419
WIT: P=0.529412 R=0.600000 F1=0.562500
AUTH: P=0.333333 R=0.440000 F1=0.379310
COURT: P=0.782609 R=0.915254 F1=0.843750
STAT: P=0.576923 R=0.582524 F1=0.579710
PREC: P=0.582609 R=0.426752 F1=0.492647
DATE: P=0.909871 R=0.909871 F1=0.909871
CASENO: P=0.807018 R=0.657143 F1=0.724409
Strict Macro-F1:        0.549252
Strict Macro-Precision: 0.561039
Strict Macro-Recall:    0.558600


Validation comparison
Original adapters: 0.4589449565809535
Adapters + CRF:    0.5492515376309127
Difference:        0.0903065810499592
APP          0.2115 -> 0.3908
RESP         0.1667 -> 0.2712
A.COUNSEL    0.3256 -> 0.2716
R.COUNSEL    0.4819 -> 0.4878
JUDGE        0.6269 -> 0.6774
WIT          0.2941 -> 0.5625
AUTH         0.3621 -> 0.3793
COURT        0.7737 -> 0.8438
ST

## Compare validation scores

In [ ]:
for method in ('full','adapters','lorma'):
    path=COLLECTION/method/'best_validation_metrics.json'
    print(method,read_json(path)['macro_f1'] if path.exists() else 'not evaluated')